In [1]:
import pandas as pd
import numpy as np
from functions.running import prepare_data
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.decomposition import PCA
from functions.training import train
from functions.networks import SimpleNN, FullNN
import shap

In [2]:
import numpy as np

import numpy as np
import pandas as pd

def generate_synthetic_data(
    n_samples=2000,
    n_features=20,
    seed=42,
    important_features=[2, 5, 7],
    save_csv=True,
    filename="synthetic_dataset.csv"
):
    np.random.seed(seed)

    # 1. Features: uniform in [0, 1000]
    X = np.random.uniform(0, 1000, size=(n_samples, n_features))

    # 2. Geometric weights for important features
    k = len(important_features)
    base = np.array([0.3 / (2**i) for i in range(k)], dtype=float)
    weights = base / base.sum()  # normalize to sum=1

    # 3. Compute additive response
    y_signal = X[:, important_features] @ weights

    # 4. Nonlinear transformation to [0,1]
    y_cont = np.sin(y_signal / 200) * 0.5 + 0.5

    # 5. Convert to binary labels 0/1
    y = np.where(y_cont < 0.6, 0, 1)

    # 6. Save to CSV if requested
    filename = f"synthetic_dataset_{important_features}.csv"
    if save_csv:
        df = pd.DataFrame(X, columns=[f"feat_{i}" for i in range(n_features)])
        df["label"] = y
        df.to_csv(filename, index=False)
        print(f"Synthetic dataset saved to '{filename}'")

    return X, y, important_features, weights

# -------------------------------
# Evaluate recovery of ground-truth features
# -------------------------------
def evaluate_recovery(shap_values, important_features, top_k=None):
    """
    Evaluate recovery of ground-truth features.
    
    shap_values: 
        - list of arrays (multi-class), each (n_samples, n_features)
        - or array (regression) (n_samples, n_features)
    important_features: list/array of ground-truth feature indices
    top_k: number of top features to consider
    """
    import numpy as np

    # 1. Handle multi-class SHAP values
    if isinstance(shap_values, list):
        # Each element: (n_samples, n_features)
        # Take mean absolute SHAP per feature, then average across classes
        mean_attr = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
    else:
        mean_attr = np.abs(shap_values).mean(axis=0)

    if top_k is None:
        top_k = len(important_features)
    print(mean_attr)

    mean_attr_1d = mean_attr.mean(axis=1)  # or np.max(axis=1)

    # 2. Top-k features (indices)
    top_features = np.argsort(mean_attr_1d)[-top_k:]  # 1D array of indices

    # 3. Compute recovery
    recovered = len(set(top_features) & set(important_features)) / len(important_features)

    return recovered, top_features


### PCsINIT

In [23]:
seeds = [3, 4, 5, 6, 7]
recovery_scores = []
for seed in seeds:
    important_features = np.random.choice(range(0, 20), size=seed, replace=False)
    print(important_features)
    X, y, important_features, weights = generate_synthetic_data(seed=seed, important_features=important_features)
    dataname = 'synthetic'
    input_dim = X.shape[1]
    epochs = 200
    batch_size = 64
    n_layer = 5
    init_type = 'he'
    dataname_ = dataname + str(n_layer) + init_type
    missing = False
    variance_retained = .95
    criterion = nn.CrossEntropyLoss()
    learning_rate = 0.01
    n_frozen_epochs = 30

    X_train, X_test, y_train, y_test = prepare_data(X,y, missing = missing, random_state=1461)
    input_dim = X_train.shape[1]  # Number of features

    output_dim = len(np.unique(y_train))  # Number of classes (for Iris dataset)

    pca = PCA(n_components=variance_retained)
    pca.fit(X_train)
    n_components = pca.n_components_

    hidden_dim = n_components  # Hidden layer size

    other_layers = SimpleNN(input_dim=n_components, hidden_dim=hidden_dim, output_dim=output_dim, n_layer = n_layer, init_type = init_type)

    train_loader = torch.utils.data.DataLoader(list(zip(X_train, y_train)), batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(list(zip(X_test, y_test)), batch_size=batch_size, shuffle=False)


    print("Training with PCA-initialized NN...")
    pca_init_nn = FullNN(input_dim, n_components, other_layers, activation='none', init_type=init_type)
    pca_init_nn.init_pca_weights(X_train)  # Initialize weights with PCA components

    # train on everything except the first layer
    optimizer = optim.Adam([{'params': param} for name, param in pca_init_nn.named_parameters() if not name.startswith('fc1')],
                            lr=learning_rate)

    model_pcsinit, train_losses_pcinit, test_accuracies_pcinit, training_time_pcinit, test_probs_pcinit = train(
            pca_init_nn, train_loader, test_loader, criterion, optimizer, epochs=n_frozen_epochs
        )

    # train the complete network
    optimizer = optim.Adam(pca_init_nn.parameters(), lr=learning_rate)
    model_pcsinit2, train_losses_pcinit2, test_accuracies_pcinit2, training_time_pcinit2, test_probs_pcinit2 = train(pca_init_nn, train_loader, test_loader, criterion, optimizer, epochs=epochs-n_frozen_epochs)
    train_losses_pcinit = np.concatenate([train_losses_pcinit,train_losses_pcinit2])
    test_accuracies_pcinit = np.concatenate([test_accuracies_pcinit, test_accuracies_pcinit2])
    training_time_pcinit = np.concatenate([training_time_pcinit, training_time_pcinit2])
    test_probs_pcinit = np.concatenate([test_probs_pcinit, test_probs_pcinit2])

    # ====== Interpret Full Model After Training ======
    model_final = model_pcsinit2  # model đã fine-tune cả fc1
    model_final.eval()
    print("\n=== SHAP PCSINIT ===")
    background_np = X_train.detach().cpu().numpy() if torch.is_tensor(X_train) else X_train

    # Ensure test data is NumPy array
    X_test_np = X_test.detach().cpu().numpy() if torch.is_tensor(X_test) else X_test

    def pcsinit_torch_predict(x_numpy):
        x_torch = torch.tensor(x_numpy, dtype=torch.float32)
        with torch.no_grad():
            return model_final(x_torch).numpy()

    pcsinit_explainer = shap.KernelExplainer(pcsinit_torch_predict, background_np)
    pcsinit_shap_values = pcsinit_explainer.shap_values(X_test_np[:50])

    recovery_score, top_features = evaluate_recovery(pcsinit_shap_values, important_features)
    print(f"Seed: {seed}")
    print(f"Ground-truth important features: {important_features}")
    print(f"Top recovered features by SHAP: {top_features}")
    print(f"Recovery score: {recovery_score*100:.2f}%")
    
    recovery_scores.append(recovery_score)


[14  7 11]
Synthetic dataset saved to 'synthetic_dataset_[14  7 11].csv'
Training with PCA-initialized NN...
Number of PCA components: 19
Epoch 1/30, Training Loss: 0.5851, Testing Accuracy: 0.8017, Training Time: 0.0884
Epoch 2/30, Training Loss: 0.3711, Testing Accuracy: 0.8300, Training Time: 0.1485
Epoch 3/30, Training Loss: 0.3355, Testing Accuracy: 0.8367, Training Time: 0.2146
Epoch 4/30, Training Loss: 0.2951, Testing Accuracy: 0.8433, Training Time: 0.2778
Epoch 5/30, Training Loss: 0.2799, Testing Accuracy: 0.8333, Training Time: 0.3555
Epoch 6/30, Training Loss: 0.2782, Testing Accuracy: 0.8483, Training Time: 0.4096
Epoch 7/30, Training Loss: 0.2710, Testing Accuracy: 0.8417, Training Time: 0.4745
Epoch 8/30, Training Loss: 0.2543, Testing Accuracy: 0.8450, Training Time: 0.5333
Epoch 9/30, Training Loss: 0.2489, Testing Accuracy: 0.8400, Training Time: 0.5895
Epoch 10/30, Training Loss: 0.2393, Testing Accuracy: 0.8467, Training Time: 0.6449
Epoch 11/30, Training Loss: 0.2

Using 1400 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


Epoch 168/170, Training Loss: 0.0000, Testing Accuracy: 0.9483, Training Time: 9.8239
Epoch 169/170, Training Loss: 0.0000, Testing Accuracy: 0.9483, Training Time: 9.8753
Epoch 170/170, Training Loss: 0.0000, Testing Accuracy: 0.9483, Training Time: 9.9302

=== SHAP PCSINIT ===


  0%|          | 0/50 [00:00<?, ?it/s]

[[ 2.4752621   3.0909398 ]
 [ 1.83208189  2.35637339]
 [ 1.17661348  1.46731736]
 [ 1.55616513  1.80683951]
 [ 2.43193078  2.82567161]
 [ 4.36586075  5.28316417]
 [ 2.81150143  3.35846006]
 [15.40269992 20.92748851]
 [ 0.66401734  0.90308547]
 [ 3.37728032  4.00601923]
 [ 1.75310612  2.20597479]
 [ 7.41179385  9.89008102]
 [ 1.25152613  1.53368811]
 [ 3.86085149  4.63298506]
 [36.75047963 47.50490184]
 [ 1.46889126  1.78521977]
 [ 2.27326366  2.71739187]
 [ 4.87595343  5.7731596 ]
 [ 1.50114827  1.89835657]
 [ 4.56409553  5.55414613]]
Seed: 3
Ground-truth important features: [14  7 11]
Top recovered features by SHAP: [11  7 14]
Recovery score: 100.00%
[17  9  2  1]
Synthetic dataset saved to 'synthetic_dataset_[17  9  2  1].csv'
Training with PCA-initialized NN...
Number of PCA components: 19
Epoch 1/30, Training Loss: 0.6054, Testing Accuracy: 0.7133, Training Time: 0.0909
Epoch 2/30, Training Loss: 0.4348, Testing Accuracy: 0.8417, Training Time: 0.1630
Epoch 3/30, Training Loss: 0.3

...

[[ 2.26453686  2.21760133]
 [ 1.25299281  1.3592387 ]
 [ 1.42306723  1.44782497]
 [ 1.45009754  1.48302093]
 [ 1.18437892  1.29493886]
 [ 1.32276759  1.44291763]
 [ 8.37932464  8.89478986]
 [ 0.73117744  0.77426003]
 [ 3.57272084  3.73912594]
 [ 1.6907848   1.78428517]
 [ 1.11317059  1.20707005]
 [ 0.59802609  0.66293926]
 [ 0.6464989   0.70157265]
 [ 2.73927375  3.04842157]
 [ 1.354847    1.30621927]
 [ 0.69676263  0.69693256]
 [ 0.834878    0.86773747]
 [17.36551779 18.61159327]
 [ 0.63808661  0.6355306 ]
 [ 2.66528671  2.69443554]]
Seed: 6
Ground-truth important features: [17  6 13 19  1 18]
Top recovered features by SHAP: [ 0 19 13  8  6 17]
Recovery score: 66.67%
[ 4  8 17  1  2 16 19]
Synthetic dataset saved to 'synthetic_dataset_[ 4  8 17  1  2 16 19].csv'
Training with PCA-initialized NN...
Number of PCA components: 19
Epoch 1/30, Training Loss: 0.6065, Testing Accuracy: 0.7017, Training Time: 0.1288
Epoch 2/30, Training Loss: 0.4582, Testing Accuracy: 0.8167, Training Time: 0.

Using 1400 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


Epoch 168/170, Training Loss: 0.0000, Testing Accuracy: 0.9583, Training Time: 10.5092
Epoch 169/170, Training Loss: 0.0000, Testing Accuracy: 0.9583, Training Time: 10.5647
Epoch 170/170, Training Loss: 0.0000, Testing Accuracy: 0.9583, Training Time: 10.6169

=== SHAP PCSINIT ===


  0%|          | 0/50 [00:00<?, ?it/s]

[[ 0.6498826   0.92103243]
 [ 3.22099278  4.35476168]
 [ 1.57332297  2.31267046]
 [ 0.59083526  0.94217561]
 [23.03547771 33.90103311]
 [ 0.84198661  1.47658906]
 [ 0.76981971  1.15788633]
 [ 0.7429665   1.33384918]
 [11.40813167 15.42546145]
 [ 0.7382517   1.24213361]
 [ 1.77252935  3.33829505]
 [ 1.06139798  2.02842753]
 [ 0.32766168  0.42648207]
 [ 1.1889644   2.13858814]
 [ 0.79272497  1.33895002]
 [ 1.44883693  2.69566849]
 [ 2.20290917  3.86783962]
 [ 5.86424288  7.6177289 ]
 [ 0.40798112  0.52488652]
 [ 0.95948407  1.5797424 ]]
Seed: 7
Ground-truth important features: [ 4  8 17  1  2 16 19]
Top recovered features by SHAP: [15 10 16  1 17  8  4]
Recovery score: 71.43%


In [24]:
print("Recovery scores per run:", recovery_scores)
print("Average recovery:", np.mean(recovery_scores))

Recovery scores per run: [1.0, 1.0, 0.8, 0.6666666666666666, 0.7142857142857143]
Average recovery: 0.836190476190476


### NN

In [12]:
important_features_list = [[14,7,11],[17,9,2,1],[17,2,0,9,8],[17,6,13,19,1,18],[4,8,17,1,2,16,19]]
nn_recovery_scores = []

for important_features in important_features_list: 
    
    data = pd.read_csv(f'/Volumes/Macintosh HD/Projects/PAPER/12.PCSINIT/synthetic_dataset_{important_features}.csv')
    X = data.drop(['label'], axis=1).to_numpy()
    y = data['label'].to_numpy()
    X_train, X_test, y_train, y_test = prepare_data(X,y, missing = False, random_state=len(important_features))
    dataname = 'synthetic'
    init_type = 'he'
    dataname_ = dataname + str(n_layer) + init_type
    variance_retained = .95
    criterion = nn.CrossEntropyLoss()
    learning_rate = 0.01
    n_frozen_epochs = 30
    epochs = 200
    batch_size = 64
    n_layer = 5
    input_dim = X_train.shape[1]  # Number of features

    output_dim = len(np.unique(y_train))  # Number of classes (for Iris dataset)

    pca = PCA(n_components=variance_retained)
    pca.fit(X_train)
    n_components = pca.n_components_

    hidden_dim = n_components  # Hidden layer size

    other_layers = SimpleNN(input_dim=n_components, hidden_dim=hidden_dim, output_dim=output_dim, n_layer = n_layer, init_type = init_type)

    train_loader = torch.utils.data.DataLoader(list(zip(X_train, y_train)), batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(list(zip(X_test, y_test)), batch_size=batch_size, shuffle=False)

    relu_nn =  FullNN(input_dim, n_components, other_layers, activation='none', init_type=init_type)
    optimizer = optim.Adam(relu_nn.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    nn_model, train_losses_fnn, test_accuracies_fnn, training_time_fnn, _ = train(relu_nn, train_loader, test_loader, criterion, optimizer, epochs=epochs)

    # ====== Interpret Full Model After Training ======
    model_final = nn_model 
    model_final.eval()

    # Ensure test data is NumPy array
    X_test_np = X_test.detach().cpu().numpy() if torch.is_tensor(X_test) else X_test
    background_np = X_train.detach().cpu().numpy() if torch.is_tensor(X_train) else X_train


    def nn_torch_predict(x_numpy):
        x_torch = torch.tensor(x_numpy, dtype=torch.float32)
        with torch.no_grad():
            return nn_model(x_torch).numpy()

    nn_explainer = shap.KernelExplainer(nn_torch_predict, background_np)
    nn_shap_values = nn_explainer.shap_values(X_test_np[:50])

    recovery_score, top_features = evaluate_recovery(nn_shap_values, important_features)
    print(f"Ground-truth important features: {important_features}")
    print(f"Top recovered features by SHAP: {top_features}")
    print(f"Recovery score: {recovery_score*100:.2f}%")
    
    nn_recovery_scores.append(recovery_score)


Epoch 1/200, Training Loss: 0.7608, Testing Accuracy: 0.6667, Training Time: 0.0771
Epoch 2/200, Training Loss: 0.5099, Testing Accuracy: 0.8250, Training Time: 0.1496
Epoch 3/200, Training Loss: 0.2503, Testing Accuracy: 0.9050, Training Time: 0.2157
Epoch 4/200, Training Loss: 0.1334, Testing Accuracy: 0.9233, Training Time: 0.4481
Epoch 5/200, Training Loss: 0.0884, Testing Accuracy: 0.9467, Training Time: 0.5749
Epoch 6/200, Training Loss: 0.0407, Testing Accuracy: 0.9483, Training Time: 0.6756
Epoch 7/200, Training Loss: 0.0362, Testing Accuracy: 0.9567, Training Time: 0.7824
Epoch 8/200, Training Loss: 0.0218, Testing Accuracy: 0.9650, Training Time: 0.8381
Epoch 9/200, Training Loss: 0.0154, Testing Accuracy: 0.9650, Training Time: 0.8896
Epoch 10/200, Training Loss: 0.0104, Testing Accuracy: 0.9550, Training Time: 0.9433
Epoch 11/200, Training Loss: 0.0245, Testing Accuracy: 0.9667, Training Time: 0.9967
Epoch 12/200, Training Loss: 0.0178, Testing Accuracy: 0.9600, Training Ti

Using 1400 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


Epoch 199/200, Training Loss: 0.0000, Testing Accuracy: 0.9633, Training Time: 11.6148
Epoch 200/200, Training Loss: 0.0000, Testing Accuracy: 0.9633, Training Time: 11.6698


  0%|          | 0/50 [00:00<?, ?it/s]

[[ 0.45240716  0.66520215]
 [ 0.34533931  0.59389758]
 [ 0.41477806  0.76518256]
 [ 0.57830803  0.9327212 ]
 [ 0.52770693  0.97583067]
 [ 0.5344349   0.92814274]
 [ 0.67867211  1.11377442]
 [ 7.48270925 11.52669102]
 [ 0.32156523  0.50716385]
 [ 0.45061558  0.7337166 ]
 [ 0.90564253  1.55615951]
 [ 3.71348673  5.63470592]
 [ 0.50777974  0.80841528]
 [ 0.81667657  1.28391063]
 [14.85396652 23.14161649]
 [ 0.53651969  0.83059563]
 [ 1.24869323  2.04272559]
 [ 1.03182108  1.62354461]
 [ 0.74528948  1.03864719]
 [ 0.27247199  0.43193656]]
Ground-truth important features: [14, 7, 11]
Top recovered features by SHAP: [11  7 14]
Recovery score: 100.00%
Epoch 1/200, Training Loss: 0.6592, Testing Accuracy: 0.8383, Training Time: 0.0691
Epoch 2/200, Training Loss: 0.2687, Testing Accuracy: 0.9250, Training Time: 0.1304
Epoch 3/200, Training Loss: 0.1188, Testing Accuracy: 0.9333, Training Time: 0.1846
Epoch 4/200, Training Loss: 0.0702, Testing Accuracy: 0.9433, Training Time: 0.2307
Epoch 5/200

...

[[ 1.84166242  2.62219764]
 [ 0.67422412  0.65104831]
 [ 0.73992818  0.87577258]
 [ 1.23593155  1.68576147]
 [ 1.03309063  1.28520143]
 [ 0.77691047  1.06668639]
 [ 7.55143095  7.7600209 ]
 [ 1.34759388  1.67132452]
 [ 1.15429727  1.80018562]
 [ 0.86702945  1.04521806]
 [ 0.73877651  0.92927667]
 [ 0.51177693  0.60300719]
 [ 0.93890663  1.40131166]
 [ 3.03782724  3.08461988]
 [ 0.9134147   1.16257944]
 [ 0.42327441  0.44517898]
 [ 0.75465899  0.93578358]
 [16.23680993 18.29569682]
 [ 0.29407902  0.36588914]
 [ 1.81558322  1.59651579]]
Ground-truth important features: [17, 6, 13, 19, 1, 18]
Top recovered features by SHAP: [ 7 19  0 13  6 17]
Recovery score: 66.67%
Epoch 1/200, Training Loss: 0.6321, Testing Accuracy: 0.7067, Training Time: 0.0687
Epoch 2/200, Training Loss: 0.3752, Testing Accuracy: 0.8983, Training Time: 0.1203
Epoch 3/200, Training Loss: 0.2124, Testing Accuracy: 0.8917, Training Time: 0.1791
Epoch 4/200, Training Loss: 0.1380, Testing Accuracy: 0.9267, Training Time:

Using 1400 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


Epoch 199/200, Training Loss: 0.0000, Testing Accuracy: 0.9383, Training Time: 11.0570
Epoch 200/200, Training Loss: 0.0000, Testing Accuracy: 0.9383, Training Time: 11.1184


  0%|          | 0/50 [00:00<?, ?it/s]

[[ 0.6801071   0.33687948]
 [ 5.68955837  2.97828114]
 [ 0.73958963  0.50897119]
 [ 1.31143956  0.70995354]
 [33.19443202 18.05928293]
 [ 2.45170344  1.16472348]
 [ 0.69656222  0.40657855]
 [ 1.02015342  0.65531501]
 [17.39014274  9.4756445 ]
 [ 1.29127557  0.60479933]
 [ 0.40728301  0.23984246]
 [ 1.24167589  0.63657488]
 [ 1.05479092  0.60521433]
 [ 1.29044599  0.69149542]
 [ 1.86315115  0.91776085]
 [ 0.81313659  0.47539081]
 [ 1.19483232  0.60552159]
 [ 8.99495649  4.8804259 ]
 [ 0.50850174  0.36031844]
 [ 0.89425531  0.52746346]]
Ground-truth important features: [4, 8, 17, 1, 2, 16, 19]
Top recovered features by SHAP: [ 3 14  5  1 17  8  4]
Recovery score: 57.14%


In [13]:
print("NN Recovery scores per run:", nn_recovery_scores)
print("NN Average recovery:", np.mean(nn_recovery_scores))

NN Recovery scores per run: [1.0, 1.0, 0.8, 0.6666666666666666, 0.5714285714285714]
NN Average recovery: 0.8076190476190476
